In [1]:
import numpy as np
from skrebate import ReliefF
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, roc_curve, average_precision_score, 
                             cohen_kappa_score, matthews_corrcoef, balanced_accuracy_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from joblib import Parallel, delayed
import multiprocessing

import os

# Limit each parallel process to one thread per library
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['TORCH_NUM_THREADS'] = '1'  # For PyTorch

# Load the dataset
data = pd.read_excel("class123_dataset.xlsx")

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Apply ReliefF for feature selection
relief = ReliefF()
relief.fit(X.values, Y.values)

# Get the feature importances
feature_importances = relief.feature_importances_

# Sort features by importance (descending order)
sorted_indices = np.argsort(feature_importances)[::-1]
sorted_features = [(index, feature_importances[index]) for index in sorted_indices]

# Define classifiers
classifiers = [
    ('Decision Tree', DecisionTreeClassifier()),
    ('Random Forest', RandomForestClassifier()),
    ('SVM', SVC(probability=True, random_state=42)),  # Added random_state for reproducibility
    ('KNN', KNeighborsClassifier()),
    ('Naive Bayes', GaussianNB()),
    ('AdaBoost', AdaBoostClassifier(random_state=42)),
    ('Gradient Boosting', GradientBoostingClassifier(random_state=42)),
    ('MLP', MLPClassifier(max_iter=1000, random_state=42))  # Ensured reproducibility
]

# Initialize a dictionary to store the best AUC and feature combination for each algorithm
best_results_by_classifier = {
    'Decision Tree': (0, None, None),  # (AUC, feature_indices, fold)
    'Random Forest': (0, None, None),
    'SVM': (0, None, None),
    'KNN': (0, None, None),
    'Naive Bayes': (0, None, None),
    'AdaBoost': (0, None, None),
    'Gradient Boosting': (0, None, None),
    'MLP': (0, None, None)
}

# Define Stratified K-Fold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

def evaluate_classifiers(num_features):
    """
    Evaluate all classifiers using the top `num_features` features.

    Returns a list of tuples:
    (classifier_name, avg_auc, selected_features_indices, best_fold)
    """
    results = []
    selected_features_indices = [index for index, _ in sorted_features[:num_features]]
    X_reduced = X.iloc[:, selected_features_indices]

    for name, model in classifiers:
        aucs = []
        best_fold = -1
        fold_num = 0
        for train_index, test_index in skf.split(X_reduced, Y):
            X_train_fold, X_test_fold = X_reduced.iloc[train_index], X_reduced.iloc[test_index]
            y_train_fold, y_test_fold = Y.iloc[train_index], Y.iloc[test_index]

            # Train the model on the current fold
            model.fit(X_train_fold, y_train_fold)

            # Get predicted probabilities for AUC calculation
            if hasattr(model, "predict_proba"):
                y_pred_probs_fold = model.predict_proba(X_test_fold)[:, 1]
            else:
                # For models that don't have predict_proba, use decision function or predictions
                if hasattr(model, "decision_function"):
                    y_pred_probs_fold = model.decision_function(X_test_fold)
                    # Some models return shape (n_samples,), ensure it's positive for AUC
                    y_pred_probs_fold = (y_pred_probs_fold - y_pred_probs_fold.min()) / (y_pred_probs_fold.max() - y_pred_probs_fold.min())
                else:
                    y_pred_probs_fold = model.predict(X_test_fold)

            # Calculate AUC for the current fold and store
            try:
                auc = roc_auc_score(y_test_fold, y_pred_probs_fold)
            except ValueError:
                # Handle cases where only one class is present in y_test_fold
                auc = 0.5  # Neutral AUC

            aucs.append(auc)

            # Track the fold with the highest AUC
            if auc > best_results_by_classifier[name][0]:
                best_fold = fold_num

            fold_num += 1

        # Calculate average AUC across all folds
        avg_auc = np.mean(aucs)

        results.append((name, avg_auc, selected_features_indices, best_fold))

    return results

# Parallelize the evaluation over different numbers of features
all_results = Parallel(n_jobs=26)(delayed(evaluate_classifiers)(num_features) for num_features in range(1, 258))

# Process the results to find the best AUC for each classifier
for feature_set in all_results:
    for name, avg_auc, selected_features_indices, best_fold in feature_set:
        if avg_auc > best_results_by_classifier[name][0]:
            best_results_by_classifier[name] = (avg_auc, selected_features_indices, best_fold)

# Display the best results
for classifier, (auc, features, fold) in best_results_by_classifier.items():
    feature_names = X.columns[features].tolist() if features else []
    print(f"Classifier: {classifier}")
    print(f"  Best AUC: {auc:.4f}")
    print(f"  Number of Features: {len(features)}")
    print(f"  Feature Indices: {features}")
    print(f"  Feature Names: {feature_names}")
    print(f"  Best Fold: {fold}")
    print("-" * 50)

Classifier: Decision Tree
  Best AUC: 0.7276
  Number of Features: 60
  Feature Indices: [28, 234, 36, 215, 189, 37, 77, 26, 250, 188, 236, 181, 70, 55, 48, 67, 202, 244, 78, 65, 52, 237, 211, 57, 185, 229, 231, 219, 62, 220, 51, 200, 30, 242, 233, 198, 34, 25, 212, 205, 252, 32, 29, 248, 253, 75, 66, 247, 56, 58, 12, 221, 41, 197, 7, 63, 217, 1, 251, 199]
  Feature Names: ['rs591058', 'rs1800797', 'rs13946', 'rs25487', 'rs2228570', 'rs12722', 'rs2104772', 'rs1144393', 'rs3196378', 'rs1544410', 'rs2237352', 'rs1137101', 'rs10263021', 'rs4454832', 'rs7528684', 'rs2234693', 'rs10132091', 'rs7035322', 'rs1330363', 'rs1011814', 'rs1249269', 'rs4725069', 'rs820218', 'rs62051384', 'rs11232681', 'rs4701616', 'rs1800629', 'rs11177', 'rs143383', 'rs6617', 'rs6481512', 'rs17756404', 'rs4986938', 'rs13317', 'rs4730153', 'rs911263', 'rs970547', 'rs11225395', 'rs3018362', 'rs11629171', 'rs10992075', 'rs4789932', 'rs2252070', 'rs3789870', 'rs1590', 'rs42531', 'rs11154027', 'rs10759753', 'rs2761884',